In [0]:
%pip install statsmodels

In [0]:
dbutils.fs.rm("abfss://refinedzone@dls2adseus2edhprd01.dfs.core.windows.net/value_stream_monitoring/databricks_costs_archive_staging", recurse=True)
spark.sql("DROP TABLE IF EXISTS prd.value_stream_monitoring.databricks_costs_archive_staging")
dbutils.fs.rm("abfss://refinedzone@dls2adseus2edhprd01.dfs.core.windows.net/value_stream_monitoring/databricks_costs_archive", recurse=True)
spark.sql(f"DROP TABLE IF EXISTS prd.value_stream_monitoring.databricks_costs_archive")

In [0]:
from datetime import datetime, timedelta, timezone
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, DateType
import pyspark.sql.functions as F
from pyspark.sql.functions import element_at, split, col
from pyspark.sql.functions import col, to_date

schema = StructType([
    StructField("UsageDate", DateType(), True),
    StructField("Meter", StringType(), True),
    StructField("ServiceName", StringType(), True),
    StructField("ResourceId", StringType(), True),
    StructField("Cost", DoubleType(), True)
])

table_name = "prd.value_stream_monitoring.databricks_costs_archive"
staging_name = "prd.value_stream_monitoring.databricks_costs_archive_staging"
delta_path = "abfss://refinedzone@dls2adseus2edhprd01.dfs.core.windows.net/value_stream_monitoring/databricks_costs_archive"
staging_path = "abfss://refinedzone@dls2adseus2edhprd01.dfs.core.windows.net/value_stream_monitoring/databricks_costs_archive_staging"

try:
    df1 = spark.read.format("csv").option("header", "true").schema(schema).load(
        f"abfss://databricks@sasscaleservesavecusprd.dfs.core.windows.net/databricks_meter_cost_20260801.csv"
    )
except Exception as e:
    print(f"Read failed: {e}")
    df1 = None


spark.conf.set("spark.sql.shuffle.partitions", "10")

if df1 is None:
    df = None
    dbutils.notebook.exit("WARNING: No cost files found")
else:
    df = df1
    df=df.groupBy("UsageDate", "Meter").agg(F.sum("Cost").alias("Cost"))
    df = df.withColumn("UsageDate", to_date(col("UsageDate"), "yyyy-MM-dd"))
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("path", staging_path) \
        .saveAsTable(staging_name)
    if not spark.catalog.tableExists(table_name):
        spark.sql(f"""
            CREATE TABLE {table_name}
            USING DELTA
            LOCATION '{delta_path}'
            AS SELECT * FROM {staging_name}
        """)
    else:
        spark.sql(f"""
            MERGE INTO {table_name} AS tgt
            USING {staging_name} AS src
            ON tgt.UsageDate = src.UsageDate
               AND tgt.Meter = src.Meter
            WHEN MATCHED THEN UPDATE SET tgt.Cost = src.Cost
            WHEN NOT MATCHED THEN INSERT *
        """)


    spark.catalog.dropTempView(staging_name)

    spark.sql(f"DROP TABLE IF EXISTS {staging_name}")

In [0]:
import pandas as pd

dfCost = spark.sql("""
    WITH usage_agg AS (
        SELECT usage_date, sku_name, SUM(usage_quantity) AS total_usage
        FROM system.billing.usage
        GROUP BY usage_date, sku_name
    ),
    license_new AS (
        SELECT Date AS UsageDate, Meter, Cost
        FROM value_stream_monitoring.databricks_costs_daily
    ),
    license_old AS (
        SELECT a.UsageDate, a.Meter, a.Cost
        FROM value_stream_monitoring.databricks_costs_archive a
        LEFT ANTI JOIN (SELECT DISTINCT UsageDate FROM license_new) n
            ON a.UsageDate = n.UsageDate
    ),
    cost_agg AS (
        SELECT c.UsageDate, m.sku, SUM(c.Cost) AS total_cost
        FROM (SELECT * FROM license_new UNION ALL SELECT * FROM license_old) c
        JOIN value_stream_monitoring.sku_meter_mapping m ON c.Meter = m.meter
        GROUP BY c.UsageDate, m.sku
    ),
    infra_new AS (
        SELECT Date AS UsageDate, Cost
        FROM value_stream_monitoring.databricks_infra_costs_daily
    ),
    infra_old AS (
        SELECT a.UsageDate, a.Cost
        FROM value_stream_monitoring.databricks_infra_costs_archive a
        LEFT ANTI JOIN (SELECT DISTINCT UsageDate FROM infra_new) n
            ON a.UsageDate = n.UsageDate
    ),
    infra_agg AS (
        SELECT UsageDate, SUM(Cost) AS total_cost
        FROM (SELECT * FROM infra_new UNION ALL SELECT * FROM infra_old)
        GROUP BY UsageDate
    ),
    storage_agg AS (
        SELECT CAST(UsageDate AS DATE) AS UsageDate, SUM(Cost) AS total_cost
        FROM value_stream_monitoring.databricks_storage_costs_archive
        GROUP BY CAST(UsageDate AS DATE)
    ),
    user_agg AS (
        WITH notebook_users AS (
            SELECT
                sku_name,
                usage_date,
                count(distinct identity_metadata.run_as) AS number_of_users,
                count(distinct record_id) AS total_runs
            FROM system.billing.usage
            GROUP BY usage_date, sku_name
        ),
        sql_serverless_users AS (
            SELECT
                'PREMIUM_SERVERLESS_SQL_COMPUTE_US_EAST_2' AS sku_name,
                date(qh.start_time) AS usage_date,
                count(distinct qh.executed_by) AS number_of_users,
                count(distinct statement_id) AS total_runs
            FROM system.query.history qh
            JOIN system.compute.warehouses w ON qh.compute.warehouse_id = w.warehouse_id
            WHERE w.warehouse_type = 'SERVERLESS'
            GROUP BY date(qh.start_time)
        ),
        sql_classic_users AS (
            SELECT
                'PREMIUM_SQL_COMPUTE' AS sku_name,
                date(qh.start_time) AS usage_date,
                count(distinct qh.executed_by) AS number_of_users,
                count(distinct statement_id) AS total_runs
            FROM system.query.history qh
            JOIN system.compute.warehouses w ON qh.compute.warehouse_id = w.warehouse_id
            WHERE w.warehouse_type != 'SERVERLESS'
            GROUP BY date(qh.start_time)
        )
        SELECT usage_date, sku_name, SUM(number_of_users) AS number_of_users, SUM(total_runs) AS total_runs
        FROM (
            SELECT * FROM notebook_users
            UNION ALL
            SELECT * FROM sql_serverless_users
            UNION ALL
            SELECT * FROM sql_classic_users
        )
        GROUP BY usage_date, sku_name
    )
    SELECT
        u.usage_date,
        m.sku,
        u.total_usage,
        COALESCE(c.total_cost, 0) AS total_cost,
        COALESCE(us.number_of_users, 0) AS number_of_users,
        COALESCE(us.total_runs, 0) AS total_runs
    FROM usage_agg u
    JOIN (SELECT DISTINCT sku FROM value_stream_monitoring.sku_meter_mapping) m ON u.sku_name = m.sku
    LEFT JOIN cost_agg c ON m.sku = c.sku AND u.usage_date = c.UsageDate
    LEFT JOIN user_agg us ON u.usage_date = us.usage_date AND u.sku_name = us.sku_name
    UNION ALL
    SELECT
        UsageDate AS usage_date,
        'INFRA' AS sku,
        CAST(NULL AS DOUBLE) AS total_usage,
        total_cost,
        0 AS number_of_users,
        0 AS total_runs
    FROM infra_agg
    UNION ALL
    SELECT
        UsageDate AS usage_date,
        'STORAGE' AS sku,
        CAST(NULL AS DOUBLE) AS total_usage,
        total_cost,
        0 AS number_of_users,
        0 AS total_runs
    FROM storage_agg
""").toPandas()

dfCost['usage_date'] = pd.to_datetime(dfCost['usage_date'])

dupes = dfCost[dfCost.duplicated(['usage_date', 'sku'], keep=False)]
if not dupes.empty:
    raise ValueError(f"Duplicate usage_date/sku rows:\n{dupes.sort_values(['usage_date', 'sku'])}")

costPivot = dfCost.pivot(index='usage_date', columns='sku', values='total_cost').asfreq('D').fillna(0)
userPivot = dfCost.pivot(index='usage_date', columns='sku', values='number_of_users').asfreq('D').fillna(0)
runPivot = dfCost.pivot(index='usage_date', columns='sku', values='total_runs').asfreq('D').fillna(0)

has_cost = costPivot.index[costPivot.sum(axis=1) > 0]
first, last = has_cost.min(), has_cost.max() - pd.Timedelta(days=1)
costPivot = costPivot.loc[first:last]
userPivot = userPivot.loc[first:last]
runPivot = runPivot.loc[first:last]

In [0]:
for sku in costPivot.columns:
  user_corr = costPivot[sku].corr(userPivot[sku])
  run_corr = costPivot[sku].corr(runPivot[sku])
  print(f"{sku}: run_correlation = {run_corr:.2f}, user_correlation = {user_corr:.2f}")

In [0]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

horizon = 180
forecasts = {}

def forecast_regressor(series, horizon):
    model = ExponentialSmoothing(series, trend='add', seasonal=None).fit()
    return model.forecast(horizon)

for sku in costPivot.columns:
    series = costPivot[sku]
    corr = series.corr(userPivot[sku])

    if abs(corr) > 0.75:
        future_users = forecast_regressor(userPivot[sku], horizon)
        model = SARIMAX(series, exog=userPivot[sku], order=(1,1,1),
                         seasonal_order=(1,1,1,14)).fit(disp=False)
        forecast = model.forecast(horizon, exog=future_users)
    else:
        model = ExponentialSmoothing(series, trend='add', seasonal='add', seasonal_periods=14).fit()
        forecast = model.forecast(horizon)

    forecasts[sku] = forecast

In [0]:
bottom_up_total = pd.DataFrame(forecasts).sum(axis=1)

naive_series = costPivot.sum(axis=1)
naive_model = ExponentialSmoothing(naive_series, trend='add', seasonal='add', seasonal_periods=14).fit()
naive_forecast = naive_model.forecast(horizon)

actuals = naive_series[naive_series.index.year == naive_series.index.max().year]

daily = pd.DataFrame({
    'actual': actuals,
    'bottom_up': pd.concat([actuals, bottom_up_total]),
    'naive_direct': pd.concat([actuals, naive_forecast]),
})

monthly = daily.resample('ME').sum(min_count=1)
monthly.index = monthly.index.strftime('%Y-%m')
monthly.index.name = 'month'

monthly.to_csv("/Workspace/Users/a835877@corp.alldata.net/Databricks-Costs/databricks_breakdowns/Databricks Forecast.csv", index=True)

In [0]:
import pyspark.sql.functions as F

forecast_table = "prd.value_stream_monitoring.databricks_forecast"
forecast_path = "abfss://refinedzone@dls2adseus2edhprd01.dfs.core.windows.net/value_stream_monitoring/databricks_forecast"

sdf = (
    spark.createDataFrame(monthly.reset_index())
    .withColumn("actual", F.when(F.isnan("actual"), None).otherwise(F.col("actual")))
    .withColumn("forecast_date", F.current_date())
)

sdf.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", forecast_path) \
    .saveAsTable(forecast_table)

In [0]:
import matplotlib.pyplot as plt
pos = monthly.index.get_loc(monthly['actual'].last_valid_index())
fc = monthly.iloc[pos-1:]
plt.figure(figsize=(12,5))
plt.plot(monthly.index[:pos], monthly['actual'].iloc[:pos], marker='o', label='actual')
plt.plot(fc.index, fc['bottom_up'], marker='o', label='bottom-up forecast')
plt.plot(fc.index, fc['naive_direct'], marker='o', linestyle='--', label='naive direct forecast')
plt.legend()